In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from scipy.spatial.distance import cdist
from IPython.display import HTML
import matplotlib.gridspec as gridspec
import warnings

warnings.filterwarnings("ignore")

# ==========================================
# 1. ESCHATON ENGINE CONFIGURATION
# ==========================================
plt.style.use('dark_background')
NUM_AGENTS = 550
GRID_SIZE = 160
INITIAL_ICU = 30
BASE_MORTALITY = 0.025

# States: 0:S, 1:Strain_A, 2:R, 3:Dead(Hazard), 4:Dead(Cleared), 5:Vaccinated, 6:Strain_B
COLORS = {
    0: '#00f0ff', # Cyan (Susceptible)
    1: '#ff003c', # Red (Strain A)
    2: '#00ff66', # Green (Recovered)
    3: '#ff0000', # GLOWING RED (Toxic Dead Zone)
    4: '#1a1a1a', # Dark Grey (Cleared Body)
    5: '#0055ff', # Deep Blue (Vaccinated - Absolute Immunity)
    6: '#ffaa00'  # Orange (Mutant Strain B)
}

class SentinelVEschaton:
    def __init__(self):
        # Kinematics & Demographics
        self.pos = np.random.rand(NUM_AGENTS, 3) * GRID_SIZE
        self.vel = (np.random.rand(NUM_AGENTS, 3) - 0.5) * 5
        self.states = np.zeros(NUM_AGENTS, dtype=int)
        self.timers = np.zeros(NUM_AGENTS)
        self.age_type = np.random.choice([0, 1, 2], NUM_AGENTS, p=[0.2, 0.6, 0.2])
        self.home_hub = np.random.randint(0, 3, NUM_AGENTS)
        self.hubs = np.array([[40, 40, 40], [120, 120, 120], [40, 120, 80]])

        # Meta-Systems
        self.budget = 8000.0
        self.icu_capacity = INITIAL_ICU
        self.fear_level = 0.0
        self.supply_chain = 100.0
        self.vaccine_rd = 0.0
        self.vaccine_deployed = False
        self.mutation_day = 90

        # History Logging
        self.history = {'S': [], 'A': [], 'B': [], 'D': [], 'V': [],
                        'ICU': [], 'Fear': [], 'Budget': [], 'Supply': [], 'VaxRD': []}

        # Patient Zero
        self.states[np.random.randint(NUM_AGENTS)] = 1
        self.timers[self.states == 1] = 60

    def update(self, frame):
        # --- A. MACRO ECONOMICS & R&D ---
        # Supply Chain Health (Based on healthy adults)
        working_adults = np.sum((self.age_type == 1) & ((self.states == 0) | (self.states == 2) | (self.states == 5)))
        target_supply = (working_adults / np.sum(self.age_type == 1)) * 100
        self.supply_chain += (target_supply - self.supply_chain) * 0.1 # Smooth transition

        # Income & Expansion
        if self.supply_chain > 30:
            self.budget += (working_adults * 0.15)
            if self.budget > 5000 and self.icu_capacity < 80:
                self.icu_capacity += 0.3
                self.budget -= 20.0

        # Vaccine Research Race
        if not self.vaccine_deployed and self.budget > 1000:
            self.vaccine_rd += (self.budget * 0.0001) * (self.supply_chain / 100)
            self.budget -= 10.0
            if self.vaccine_rd >= 100.0:
                self.vaccine_deployed = True

        # Deploy Vaccine (Heals and Protects 5 agents per frame once active)
        if self.vaccine_deployed:
            targets = np.where((self.states == 0) | (self.states == 2))[0]
            if len(targets) > 0:
                chosen = np.random.choice(targets, min(5, len(targets)), replace=False)
                self.states[chosen] = 5

        # Fear Mechanics
        recent_deaths = np.sum((self.states == 3) | (self.states == 4))
        self.fear_level = min(1.0, (recent_deaths / (NUM_AGENTS * 0.25)) + (1.0 - (self.supply_chain/100)))

        # Mutation Trigger
        if frame == self.mutation_day:
            potential_hosts = np.where(self.states == 0)[0]
            if len(potential_hosts) > 0:
                self.states[np.random.choice(potential_hosts, 3)] = 6 # 3 Mutant seeds

        # --- B. PHYSICS & MOVEMENT ---
        living = (self.states != 3) & (self.states != 4)

        # Hub Gravity + Fear Repulsion
        targets = self.hubs[self.home_hub]
        dir_to_hub = targets - self.pos
        dist_to_hub = np.linalg.norm(dir_to_hub, axis=1)[:, None] + 1
        pull_force = 0.2 * (1.0 - (self.fear_level * 1.5)) # High fear makes them scatter

        self.vel += (dir_to_hub / dist_to_hub) * pull_force

        # Apply velocity (Dead agents don't move)
        self.pos[living] += self.vel[living] * 0.8

        # 3D Boundary Bounce
        for dim in range(3):
            mask = (self.pos[:, dim] < 0) | (self.pos[:, dim] > GRID_SIZE)
            self.vel[mask, dim] *= -1
        self.pos = np.clip(self.pos, 0, GRID_SIZE)

        # --- C. THE TRIPLE-THREAT TRANSMISSION ---
        for threat in [1, 6, 3]: # Strain A, Strain B, Bio-Hazard (Dead Bodies)
            infected = (self.states == threat)
            susceptible = (self.states == 0)

            # Strain B ignores Recovered immunity, but Vaccinated (5) are gods.
            if threat == 6: susceptible = (self.states == 0) | (self.states == 2)

            if np.any(infected) and np.any(susceptible):
                dists = cdist(self.pos[susceptible], self.pos[infected])

                # Bio-hazards have small radius, airborne strains have large
                radius = 3.0 if threat == 3 else (6.0 if threat == 1 else 8.5)
                contact = np.any(dists < radius, axis=1)

                new_idx = np.where(susceptible)[0][contact]
                for idx in new_idx:
                    # Strain B cross-immunity check
                    if threat == 6 and self.states[idx] == 2 and np.random.rand() < 0.6: continue
                    # Apply infection (if infected by dead body, defaults to Strain A)
                    self.states[idx] = 1 if threat == 3 else threat
                    self.timers[idx] = np.random.randint(50, 100)

        # --- D. CLINICAL & SANITATION RESOLUTION ---
        current_icu_load = np.sum((self.states == 1) | (self.states == 6))

        for i in np.where(self.states != 0)[0]:
            if self.states[i] in [1, 6]: # Active Infections
                self.timers[i] -= 1
                if self.timers[i] <= 0:
                    mortality = BASE_MORTALITY * (5.0 if current_icu_load > self.icu_capacity else 1.0)
                    if self.supply_chain < 50: mortality *= 1.5 # No medicine = more death
                    if self.states[i] == 6: mortality *= 1.5

                    if np.random.rand() < mortality:
                        self.states[i] = 3 # Becomes Bio-Hazard
                        self.timers[i] = 40 # Time until sanitation clears body
                    else:
                        self.states[i] = 2

            elif self.states[i] == 3: # Bio-Hazard Cleanup
                self.timers[i] -= 1
                if self.timers[i] <= 0:
                    self.states[i] = 4 # Cleared dead body (harmless)

        # --- E. LOGGING ---
        self.history['S'].append(np.sum(self.states == 0))
        self.history['A'].append(np.sum(self.states == 1))
        self.history['B'].append(np.sum(self.states == 6))
        self.history['D'].append(np.sum((self.states == 3) | (self.states == 4)))
        self.history['V'].append(np.sum(self.states == 5))
        self.history['ICU'].append(current_icu_load)
        self.history['Fear'].append(self.fear_level * 100)
        self.history['Budget'].append(max(0, self.budget / 100))
        self.history['Supply'].append(self.supply_chain)
        self.history['VaxRD'].append(min(100.0, self.vaccine_rd))

# ==========================================
# 2. ESCHATON CINEMATIC DASHBOARD
# ==========================================
sim = SentinelVEschaton()
fig = plt.figure(figsize=(24, 14), facecolor='#050505')
gs = gridspec.GridSpec(3, 2, width_ratios=[1.5, 1])

# Left: Massive Rotating 3D Map
ax1 = fig.add_subplot(gs[:, 0], projection='3d', facecolor='#050505')

# Right: Telemetry Panels
ax2 = fig.add_subplot(gs[0, 1], facecolor='#0d1117') # Biometrics
ax3 = fig.add_subplot(gs[1, 1], facecolor='#0d1117') # Logistics
ax4 = fig.add_subplot(gs[2, 1], facecolor='#0d1117') # Medical/R&D

def animate(frame):
    sim.update(frame)
    ax1.clear(); ax1.set_axis_off()

    # 1. ORBITAL CAMERA ROTATION
    ax1.view_init(elev=20., azim=frame * 1.5) # Spins the 3D map dynamically

    # Render Agents
    ax1.scatter(sim.pos[:,0], sim.pos[:,1], sim.pos[:,2],
                c=[COLORS[s] for s in sim.states], s=40, edgecolors='white', lw=0.1)

    # Render Hubs
    ax1.scatter(sim.hubs[:,0], sim.hubs[:,1], sim.hubs[:,2], s=600, color='yellow', alpha=0.03)

    title_text = f"ESCHATON PROTOCOL | DAY {frame}\nORBITAL SURVEILLANCE ACTIVE"
    if sim.vaccine_deployed: title_text += "\n[ VACCINE DEPLOYED - CONTAINMENT IMMINENT ]"
    ax1.set_title(title_text, color='#00f0ff' if sim.vaccine_deployed else 'red', weight='bold', fontsize=14)

    # Graph 1: Bio-Trajectory
    ax2.clear()
    ax2.plot(sim.history['A'], color=COLORS[1], lw=3, label='Strain A')
    ax2.plot(sim.history['B'], color=COLORS[6], lw=3, label='Mutant Strain B')
    ax2.plot(sim.history['D'], color='#666666', lw=2, label='Deceased')
    ax2.plot(sim.history['V'], color=COLORS[5], lw=3, label='Vaccinated')
    ax2.legend(loc='upper right', frameon=False); ax2.set_title("GLOBAL BIOMETRICS", color='white')

    # Graph 2: Logistics & Fear
    ax3.clear()
    ax3.plot(sim.history['Supply'], color='#00ff66', lw=2, label='Supply Chain Integrity %')
    ax3.plot(sim.history['Fear'], color='magenta', lw=2, label='Public Panic/Fear %')
    ax3.plot(sim.history['Budget'], color='gold', lw=2, ls=':', label='Treasury (Scaled)')
    ax3.legend(loc='upper right', frameon=False); ax3.set_title("CIVILIAN INFRASTRUCTURE", color='white')
    ax3.set_ylim(0, 110)

    # Graph 3: Medical R&D
    ax4.clear()
    ax4.fill_between(range(len(sim.history['ICU'])), sim.history['ICU'], color='red', alpha=0.3)
    ax4.axhline(y=sim.icu_capacity, color='white', linestyle='--', label='Max ICU Beds')

    # Vaccine Progress Bar effect
    ax4.plot(sim.history['VaxRD'], color='#0055ff', lw=4, label='Vaccine R&D Progress %')
    ax4.legend(loc='upper left', frameon=False); ax4.set_title("MEDICAL CAPABILITIES", color='white')
    ax4.set_ylim(0, max(110, sim.icu_capacity + 20))

    if frame % 25 == 0:
        status = "DEPLOYED" if sim.vaccine_deployed else f"{sim.vaccine_rd:.1f}%"
        print(f"[ORBITAL LINK] Day {frame} | Vax R&D: {status} | Supply: {sim.supply_chain:.1f}%")

print("BOOTING ESCHATON PROTOCOL... ALL SYSTEMS GO.")
ani = FuncAnimation(fig, animate, frames=220, interval=45)
plt.close()
HTML(ani.to_html5_video())

BOOTING ESCHATON PROTOCOL... ALL SYSTEMS GO.
[ORBITAL LINK] Day 0 | Vax R&D: 0.8% | Supply: 100.0%
[ORBITAL LINK] Day 0 | Vax R&D: 1.6% | Supply: 99.9%
[ORBITAL LINK] Day 25 | Vax R&D: 22.3% | Supply: 99.3%
[ORBITAL LINK] Day 50 | Vax R&D: 42.9% | Supply: 85.9%
[ORBITAL LINK] Day 75 | Vax R&D: 58.0% | Supply: 54.7%
[ORBITAL LINK] Day 100 | Vax R&D: 66.6% | Supply: 26.2%
[ORBITAL LINK] Day 125 | Vax R&D: 71.4% | Supply: 24.3%
[ORBITAL LINK] Day 150 | Vax R&D: 77.9% | Supply: 42.0%
[ORBITAL LINK] Day 175 | Vax R&D: 88.5% | Supply: 63.4%
[ORBITAL LINK] Day 200 | Vax R&D: DEPLOYED | Supply: 76.4%
